In [1]:
from rdflib import Graph, URIRef, Literal
from rdfine import GraphReader, GraphDict, PrefixStore, parse_config
from compilers import PipelineExtractor, PipelineAssembler, LdioConfigCompiler, RdfcConfigCompiler, DockerComposeCompiler, SemanticWorksCompiler
import pandas as pd
import yaml
import json

#### Loading the graph

In [2]:
# Loading the graph
input_folder = "..\\data\\"
catalog_graph = Graph()
catalog_graph.parse(input_folder + "demo pipeline.ttl", publicID = "file:///workspace/pipeline/")
catalog_reader = GraphReader(catalog_graph)
catalog_reader = catalog_reader.infer(input_folder + "inference_rules.yaml")

#### Extracting the pipeline

In [3]:
pipeline_id = ":DemonstratorPipeline"
pipeline_graph = PipelineExtractor(pipeline_id, catalog_reader.graph).compile()

#### Assembling the pipeline build

In [4]:
build_graph = PipelineAssembler(pipeline_graph).compile()

#### Compiling the configs for the semantic works components

In [5]:
build_graph = SemanticWorksCompiler(build_graph).compile()

#### Compiling a LDIO Config file

In [6]:
print(LdioConfigCompiler(build_graph).compile())

input:
  adapter:
    name: Ldio:RdfAdapter
  name: Ldio:HttpInPoller
  config:
    cron: '*/10 * * * * *'
    url: https://dishacled-api.azurewebsites.net/api/v1/source-a/current
transformers:
- name: Ldio:SparqlConstructTransformer
  config:
    query: "\n            PREFIX dct: <http://purl.org/dc/terms/> .\n            PREFIX\
      \ prov: <http://www.w3.org/ns/prov#> .\n            CONSTRUCT {\n          \
      \    ?versionedS ?p ?o ;\n                  prov:generatedAtTime ?generatedAtTime\
      \ ;\n                  dct:isVersionOf ?s .\n            } WHERE {\n       \
      \       ?s ?p ?o .\n              BIND(URI(CONCAT(STR(?s), '/', STR(?now)))\
      \ as ?versionedS)\n              BIND (NOW() as ?generatedAtTime)\n        \
      \    }\n          "
outputs:
- name: Ldio:HttpOut



#### Compiling a RDF-Connect Config file

In [7]:
print(RdfcConfigCompiler(build_graph).compile())

@base <file:///workspace/pipeline/> .
@prefix : <http://example.org/example/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfc: <https://w3id.org/rdf-connect#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<> a rdfc:Pipeline ;
    owl:imports <file:///usr/local/lib/python3.13/site-packages/rdfc_http_in/processor.ttl>,
        <file:///usr/local/lib/python3.13/site-packages/rdfc_threshold_monitoring/processor.ttl>,
        <node_modules/@rdfc/sparql-ingest-processor-ts/processors.ttl> ;
    rdfc:consistsOf :env_1,
        :env_2 .

:HttpIn a rdfc:HttpIn ;
    :out :channel_0 .

:SparqlIngest a rdfc:SparqlIngest ;
    :in :channel_1 ;
    :ingestConfig [ :operationMode "Replication" ] .

:ThresholdMonitoring a rdfc:thresholdMonitoringProcessor ;
    :in :channel_0 ;
    :out :channel_1 ;
    :unit <http://qudt.org/vocab/unit/CentiM> ;
    :value 30 .

:env_1 rdfc:instantiates rdfc:NodeRunner ;
    rdfc:processor :SparqlIngest,
        :ThresholdMonitoring .

:env_2 

#### Compiling the Docker Compose Config file

In [8]:
print(DockerComposeCompiler(build_graph).compile())

berichtencentrum-deliver-email-service:
  image: lblod/berichtencentrum-deliver-email-service
  environment:
    EMAIL_CRON_PATTERN: '*/1 * * * * *'
    HOURS_DELIVERING_TIMEOUT: '1'
    MAILFOLDER_URI: http://data.lblod.info/id/mail-folders/2
ldio-workbench:
  container_name: ldio-workbench
  image: ldes/ldi-orchestrator:2.8.0-SNAPSHOT
  ports:
  - 8080:8080
rdf-connect:
  container_name: rdf-connect
  image: rdf-connect:latest
  build: ../../resources/rdfc-docker
  volumes:
  - ./rdfc_pipeline.ttl:/workspace/pipeline/pipeline.ttl:ro
  environment:
    LOG_LEVEL: debug
  command: npx rdfc /workspace/pipeline/pipeline.ttl
ldio-pipeline-starter:
  image: curlimages/curl
  volumes:
  - ./ldio_pipeline.yml:/pipeline.yml:ro
  command: 'sh -c " sleep 30 && curl -X POST -H ''content-type: application/yaml''
    http://ldio-workbench:8080/admin/api/v1/pipeline --data-binary @/pipeline.yml
    "'
error-alert:
  image: lblod/loket-error-alert-service
  volumes:
  - ./config/error-alert/:/config